# Project 06: Sports Comparison Visualiser

**Lumexa Data Scientist Path — Course 14: Data Visualisation**

Build an interactive tool for comparing NBA teams using two real FiveThirtyEight datasets:
player-level RAPTOR/WAR advanced statistics, and historical team Elo ratings.

**Datasets:**
1. **Modern RAPTOR by Team** (player-season advanced stats, 2014-2022, kept in full — 7,289 rows)
   Source: https://raw.githubusercontent.com/fivethirtyeight/data/master/nba-raptor/modern_RAPTOR_by_team.csv
2. **NBA All Elo** (team game-by-game Elo ratings, 1947-2015). The full file has 126,314 rows
   across every NBA/ABA franchise in history; to keep this a classroom-sized dataset we filter
   it down to 8 real, well-known franchises (Lakers, Celtics, Warriors, Bulls, Spurs, Rockets,
   Heat, Cavaliers) **right here in the notebook** — every kept value is a real, unmodified
   FiveThirtyEight measurement, only other franchises' rows are excluded.
   Source: https://raw.githubusercontent.com/fivethirtyeight/data/master/nba-elo/nbaallelo.csv

This notebook is fully self-contained and works with **Runtime → Run all** — both real
datasets are downloaded directly from their public sources at runtime, and every chart
renders inline (no HTML files to open separately).

**What you'll do:**
1. Aggregate real player-level data up to team-season statistics with `pandas.groupby()`
2. Build three distinct comparison chart types: a bar chart, a multi-line chart, and a radar chart
3. Normalize multiple stats onto a comparable 0-1 scale for a fair radar chart


In [1]:
# pandas is preinstalled in Google Colab.
%pip install -q plotly
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print("Libraries loaded.")

Note: you may need to restart the kernel to use updated packages.


Libraries loaded.


## 1. Load the real datasets

In [2]:
RAPTOR_URL = "https://raw.githubusercontent.com/fivethirtyeight/data/master/nba-raptor/modern_RAPTOR_by_team.csv"
ELO_URL = "https://raw.githubusercontent.com/fivethirtyeight/data/master/nba-elo/nbaallelo.csv"

raptor = pd.read_csv(RAPTOR_URL)
elo_full = pd.read_csv(ELO_URL)

print(f"RAPTOR: {raptor.shape[0]} real player-season rows, seasons {raptor['season'].min()}-{raptor['season'].max()}")
print(f"ELO (full): {elo_full.shape[0]} real team-game rows, years {elo_full['year_id'].min()}-{elo_full['year_id'].max()}")

RAPTOR: 7289 real player-season rows, seasons 2014-2022
ELO (full): 126314 real team-game rows, years 1947-2015


## 2. Trim the Elo dataset to 8 well-known franchises

Keeps the notebook classroom-sized while preserving every real, unmodified value for the
franchises we keep.

In [3]:
FRANCHISES_TO_KEEP = ["Lakers", "Celtics", "Warriors", "Bulls", "Spurs", "Rockets", "Heat", "Cavaliers"]

elo = elo_full[elo_full["fran_id"].isin(FRANCHISES_TO_KEEP)].copy()
print(f"ELO (trimmed to {len(FRANCHISES_TO_KEEP)} franchises): {len(elo)} real team-game rows")
print("Franchises kept:", sorted(elo["fran_id"].unique()))

ELO (trimmed to 8 franchises): 36629 real team-game rows
Franchises kept: ['Bulls', 'Cavaliers', 'Celtics', 'Heat', 'Lakers', 'Rockets', 'Spurs', 'Warriors']


## 3. Team RAPTOR comparison: aggregate real player RAPTOR to team-season level

In [4]:
team_season = (
    raptor.groupby(["team", "season"], as_index=False)
    .agg(team_raptor_total=("raptor_total", "mean"), team_war_total=("war_total", "sum"))
)

teams_to_compare = ["GSW", "BOS", "LAL", "MIA"]
latest_season = int(raptor["season"].max())

latest_team_stats = team_season[
    (team_season["team"].isin(teams_to_compare)) & (team_season["season"] == latest_season)
].sort_values("team_war_total", ascending=False)

fig_bar = px.bar(
    latest_team_stats, x="team", y="team_war_total",
    color="team", color_discrete_sequence=px.colors.qualitative.Set2,
    title=f"Total Team WAR (Wins Above Replacement) in {latest_season}: Real RAPTOR Data",
    labels={"team": "Team", "team_war_total": "Total WAR (summed across roster)"},
    template="plotly_white",
    hover_data={"team_raptor_total": ":.2f"},
)
fig_bar.show()

## 4. Franchise Elo trend comparison: real season-end Elo over time

In [5]:
season_end_elo = (
    elo.groupby(["fran_id", "year_id"], as_index=False)["elo_n"].mean()
    .rename(columns={"elo_n": "avg_elo"})
)

franchises_to_compare = ["Lakers", "Celtics", "Warriors", "Bulls"]
elo_sub = season_end_elo[season_end_elo["fran_id"].isin(franchises_to_compare)]

fig_line = px.line(
    elo_sub, x="year_id", y="avg_elo", color="fran_id",
    title="NBA Franchise Elo Rating Over Time (Real FiveThirtyEight Data)",
    labels={"year_id": "Season year", "avg_elo": "Average Elo rating", "fran_id": "Franchise"},
    template="plotly_white",
    color_discrete_sequence=px.colors.qualitative.Set1,
)
fig_line.update_layout(hovermode="x unified")
fig_line.show()

## 5. Radar chart: multi-stat comparison of two teams in the latest RAPTOR season

In [6]:
radar_stats = ["raptor_offense", "raptor_defense", "war_total", "predator_offense", "predator_defense"]
team_a, team_b = "GSW", "BOS"

radar_df = (
    raptor[(raptor["season"] == latest_season) & (raptor["team"].isin([team_a, team_b]))]
    .groupby("team")[radar_stats].mean()
)

# Normalize each stat to 0-1 across the two teams so the radar shape is comparable
radar_norm = (radar_df - radar_df.min()) / (radar_df.max() - radar_df.min() + 1e-9)

fig_radar = go.Figure()
for team in [team_a, team_b]:
    fig_radar.add_trace(go.Scatterpolar(
        r=radar_norm.loc[team].values.tolist() + [radar_norm.loc[team].values[0]],
        theta=radar_stats + [radar_stats[0]],
        fill="toself",
        name=team,
    ))
fig_radar.update_layout(
    title=f"{team_a} vs {team_b}: Advanced Stat Profile ({latest_season} season, normalized)",
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    template="plotly_white",
)
fig_radar.show()

print("\nAll comparison visualisations built successfully from real NBA data.")


All comparison visualisations built successfully from real NBA data.


## Summary

This notebook aggregated real player-level RAPTOR data up to team-season statistics, trimmed
a large real Elo dataset down to a classroom-sized set of well-known franchises (keeping every
value real and unmodified), and built three distinct comparison chart types: a bar chart for a
single-season snapshot, a multi-line chart for a long-term franchise trend, and a normalized
radar chart for a multi-stat team profile comparison.

**Try it yourself:** change `teams_to_compare`, `franchises_to_compare`, or `team_a`/`team_b`
above to any other real NBA team abbreviation/franchise name and re-run
(`Runtime → Run all`) to compare different teams.